# CND-MNE walkthrough

Two parts:

1. **Tiny bundled example** — always works, no download. Good for a first pass.
2. **Real public CND** — CNSP's Lalor Natural Speech sample (~120 MB). Subject 1 listening to an audiobook.

Run from the repo root (or from `examples/`; the first code cell finds the data either way).

```bash
uv sync --extra dev
uv pip install jupyter
uv run jupyter notebook examples/walkthrough.ipynb
```


# Part 1 — tiny bundled example

Two fake trials, four EEG channels (`Fz`, `Cz`, `Pz`, `Oz`), a speech envelope, and word onsets. The file already declares `uV`.


In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cnd_mne import inspect_cnd, read_cnd, read_cnd_mne

here = Path.cwd()
root = here if (here / "tests/data/minimal-cnd").exists() else here.parent
source = root / "tests/data/minimal-cnd"
scratch = root / "examples/_scratch"
scratch.mkdir(exist_ok=True)

print("CND folder:", source)
print(sorted(path.name for path in source.iterdir()))

## What those two files are

- `dataSub1.mat` — EEG for subject 1, already cut into trials
- `dataStim.mat` — what was happening on the same trials

`inspect` only reads MATLAB. It does not need a unit.


In [ ]:
cnd = read_cnd(source, subject=1)
summary = inspect_cnd(cnd)
print(
    json.dumps(
        {
            "n_trials": summary["n_trials"],
            "channels": summary["neural"]["n_channels"],
            "trial_shapes": summary["neural"]["trial_shapes"],
            "unit": summary["neural"]["data_unit"],
            "features": summary["stimulus"]["feature_names"],
        },
        indent=2,
    )
)

## How the converter sits in the middle

MNE wants one continuous recording (`Raw`). CND trials can be different lengths, and the speech envelope is not an EEG channel. So we do **not** dump the whole experiment into one `Raw`.

1. Read the `.mat` files (v5 or v7.3).
2. Keep a Python copy of the CND (`CNDRecording`): trials, stimulus tracks, leftover MATLAB fields.
3. Hand each trial to MNE as its own `Raw`.
4. Hang the leftover CND on `rec.cnd` so write-back still has the envelope.

This example already declares `uV`. Real public files often do not — then you must pass `neural_unit=...` (whatever the owner confirmed).


In [ ]:
rec = read_cnd_mne(source, subject=1)

print("trials:", len(rec.raws))
for i, raw in enumerate(rec.raws, start=1):
    print(
        f"  trial {i}: {raw.n_times} samples,",
        f"{raw.info['sfreq']} Hz, {raw.ch_names}",
    )

print("stimulus tracks on rec.cnd:", rec.cnd.stimulus.names)
print(
    "first trial EEG shape (channels x time):",
    rec.raws[0].get_data().shape,
)

`rec.raws[0]` is ordinary MNE. `rec.cnd` is everything else. You plot and filter the `Raw`. You need `rec.cnd` when you want MATLAB back.


## Look at the EEG

Trial 1, all four channels. This is fake data, so it will not look like a real scalp recording.


In [ ]:
raw = rec.raws[0]
data = raw.get_data()  # volts, which is what MNE stores
times = raw.times

fig, ax = plt.subplots(figsize=(8, 3))
for i, name in enumerate(raw.ch_names):
    ax.plot(times, data[i] * 1e6, label=name)  # plot in µV
ax.set_xlabel("time (s)")
ax.set_ylabel("µV")
ax.set_title("trial 1 EEG")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
spectrum = rec.raws[0].compute_psd(fmin=1, fmax=40, verbose="error")
fig = spectrum.plot(show=False, amplitude=False)
fig.suptitle("trial 1 power spectrum")
fig.tight_layout()
plt.show()

## Look at the stimulus tracks

These are **not** EEG channels. They live on `rec.cnd`. We ask for an MNE view when we want to plot them.


In [ ]:
envelope = rec.stimulus_raws("Speech Envelope")[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(envelope.times, envelope.get_data()[0])
ax.set_xlabel("time (s)")
ax.set_title("trial 1 speech envelope")
fig.tight_layout()
plt.show()

In [ ]:
words = rec.stimulus_raws("Word Onsets")[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(words.times, words.get_data()[0], drawstyle="steps-post")
ax.set_xlabel("time (s)")
ax.set_title("trial 1 word onsets (impulse track)")
fig.tight_layout()
plt.show()

ann = rec.stimulus_annotations("Word Onsets")[0]
pairs = list(zip(ann.onset, ann.description, strict=True))
print("same onsets as MNE annotations:", pairs)

## Do something in MNE, then write CND back

Filter each trial in place. Trial lengths stay the same, which is what write-back needs.

We write through `rec`, not a bare `Raw`, so the envelope comes with us.


In [ ]:
before = rec.raws[0].get_data().copy()

for trial_raw in rec.raws:
    trial_raw.filter(1.0, 15.0, verbose="error")

after = rec.raws[0].get_data()
print("trial 1 changed after filter:", not np.allclose(before, after))

out_dir = scratch / "filtered-cnd"
paths = rec.write_cnd(out_dir, subject=1, output_unit="uV", overwrite=True)
print("wrote:", paths.neural.name, "and", paths.stimulus.name)

In [ ]:
back = read_cnd_mne(scratch / "filtered-cnd", subject=1)
print("trials still:", len(back.raws))
print("envelope still there:", back.cnd.stimulus.names)
print(
    "filtered EEG survived MATLAB:",
    np.allclose(back.raws[0].get_data(), rec.raws[0].get_data(), atol=1e-12),
)

## Optional: glue trials for a continuous plot

This is **opt-in**. The joins are fake. MNE marks them. Do not treat the glued recording as one real take.


In [ ]:
continuous = rec.concatenate()
print("glued length (samples):", continuous.n_times)
print("boundary annotations:", list(continuous.annotations.description))

# Part 2 — real public CND (Lalor Natural Speech sample)

This is the CNSP tutorial sample of the Lalor / Broderick / Di Liberto natural-speech EEG set: one person listening to an audiobook.

It is **not** in git. The next cell downloads two files from [CNSP sample data](https://data.cnspworkshop.net/sampleData/LalorNatSpeech/) into `examples/_scratch/lalor-sample/` (~120 MB). Re-running the notebook reuses the cache.

What you should see after it loads:

- 20 trials, 128 BioSemi channels (`A1`…`D32`), mastoids as extra channels
- neural clock **64 Hz**, stimulus clock **128 Hz** (the full catalogue zip is 128/128; this tutorial sample is downsampled)
- features: `Speech Envelope Vectors`, `Word Onset Vectors`
- **no physical unit** in the file

I pass `neural_unit="uV"` so MNE can plot. That is a plotting assumption, not a confirmation from the authors. If someone later says these values are volts, the traces are in the wrong SI unit.


In [ ]:
SAMPLE_BASE = "https://www.data.cnspworkshop.net/sampleData/LalorNatSpeech"
sample_dir = scratch / "lalor-sample"
sample_dir.mkdir(exist_ok=True)

for name in ("dataStim.mat", "pre_dataSub1.mat"):
    dest = sample_dir / name
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"cached {name}: {dest.stat().st_size} bytes")
        continue
    url = f"{SAMPLE_BASE}/{name}"
    print("downloading", url)
    urllib.request.urlretrieve(url, dest)
    print(f"saved {name}: {dest.stat().st_size} bytes")

print("folder:", sorted(path.name for path in sample_dir.iterdir()))

The neural file is named `pre_dataSub1.mat` (preprocessed tutorial export). The converter treats that as subject 1, same as `dataSub1.mat`.


In [ ]:
lalor_cnd = read_cnd(sample_dir, subject=1)
lalor_summary = inspect_cnd(lalor_cnd)
warn_codes = sorted(
    {issue["code"] for issue in lalor_summary["validation"]["warnings"]}
)
print(
    json.dumps(
        {
            "n_trials": lalor_summary["n_trials"],
            "channels": lalor_summary["neural"]["n_channels"],
            "neural_hz": lalor_summary["neural"]["sfreq"],
            "stim_hz": lalor_summary["stimulus"]["sfreq"],
            "unit": lalor_summary["neural"]["data_unit"],
            "features": lalor_summary["stimulus"]["feature_names"],
            "chanlocs": lalor_summary["neural"]["has_channel_locations"],
            "mastoids": lalor_summary["neural"]["has_external_channels"],
            "warning_kinds": warn_codes,
        },
        indent=2,
    )
)
print("first/last trial shapes (time x channels):")
print(" ", lalor_summary["neural"]["trial_shapes"][0])
print(" ", lalor_summary["neural"]["trial_shapes"][-1])

Those warnings are the point of tolerant mode, not a failed load:

- no `dataUnit` / `cndVersion`
- neural 64 Hz vs stimulus 128 Hz
- every trial's EEG and envelope are slightly different lengths

The converter keeps both clocks. It does not resample to hide the mismatch.


In [ ]:
lalor = read_cnd_mne(sample_dir, subject=1, neural_unit="uV")
raw0 = lalor.raws[0]
print("trials:", len(lalor.raws))
print("trial 1:", raw0.n_times, "samples,", raw0.times[-1], "s")
print("channel names (BioSemi):", raw0.ch_names[:6], "...", raw0.ch_names[-3:])
print(
    "peak-to-peak of trial 1 if unit was uV: "
    f"{(raw0.get_data().max() - raw0.get_data().min()) * 1e6:.1f} uV"
)

## EEG traces (a few channels, first 10 seconds)

128 channels is too many for one axes. BioSemi labels here are `A1`… not 10-20 `Fz`.


In [ ]:
seconds = 10
n = int(seconds * raw0.info["sfreq"])
picks = [0, 31, 64, 96]  # A1, A32, C1, D1
eeg = raw0.get_data(picks=picks)[:, :n]
t = raw0.times[:n]

fig, axes = plt.subplots(len(picks), 1, figsize=(9, 6), sharex=True)
for ax, row, name in zip(axes, eeg, [raw0.ch_names[i] for i in picks], strict=True):
    ax.plot(t, row * 1e6, color="C0", linewidth=0.8)
    ax.set_ylabel(f"{name}\nµV")
axes[-1].set_xlabel("time (s)")
axes[0].set_title("trial 1 EEG, first 10 s (plotting assumption: uV)")
fig.tight_layout()
plt.show()

In [ ]:
spectrum = raw0.compute_psd(fmin=0.5, fmax=30, verbose="error")
fig = spectrum.plot(picks=picks, show=False, amplitude=False)
fig.suptitle("trial 1 PSD, four channels")
fig.tight_layout()
plt.show()

## Speech envelope and word onsets on the stimulus clock (128 Hz)


In [ ]:
env = lalor.stimulus_raws("Speech Envelope Vectors")[0]
n_env = int(10 * env.info["sfreq"])
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.plot(env.times[:n_env], env.get_data()[0, :n_env], color="C1")
ax.set_xlabel("time (s)")
ax.set_title("trial 1 speech envelope, first 10 s")
fig.tight_layout()
plt.show()
print("envelope samples vs EEG samples in trial 1:", env.n_times, raw0.n_times)

In [ ]:
onsets = lalor.stimulus_raws("Word Onset Vectors")[0]
ann = lalor.stimulus_annotations("Word Onset Vectors")[0]
early = [t for t in ann.onset if t < 10]
fig, ax = plt.subplots(figsize=(9, 2.2))
ax.plot(
    onsets.times[:n_env],
    onsets.get_data()[0, :n_env],
    drawstyle="steps-post",
    color="C2",
)
ax.set_xlabel("time (s)")
ax.set_title(f"trial 1 word onsets, first 10 s ({len(early)} in that window)")
fig.tight_layout()
plt.show()
print("word onsets in the whole trial:", len(ann))

## Filter in MNE, write CND back

Filter a **copy** of trial 1 so the original `lalor.raws` stay untouched. Then write the whole recording through the CND template.

This writes ~100 MB of MATLAB. That is expected.


In [ ]:
filtered = raw0.copy()
filtered.filter(1.0, 8.0, verbose="error")

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(t, eeg[0] * 1e6, alpha=0.5, label="unfiltered A1")
ax.plot(
    t,
    filtered.get_data(picks=[0])[0, :n] * 1e6,
    label="1–8 Hz",
)
ax.set_xlabel("time (s)")
ax.set_ylabel("µV")
ax.set_title("trial 1 channel A1, first 10 s")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Put the filtered trial back so write_cnd exports it.
lalor.raws[0].filter(1.0, 8.0, verbose="error")

lalor_out = scratch / "lalor-filtered"
paths = lalor.write_cnd(lalor_out, subject=1, output_unit="uV", overwrite=True)
print("wrote", paths.neural.name, paths.stimulus.name)

back = read_cnd_mne(lalor_out, subject=1, neural_unit="uV")
print("trials:", len(back.raws))
print("features still:", back.cnd.stimulus.names)
print(
    "filtered trial survived MATLAB:",
    np.allclose(
        back.raws[0].get_data(),
        lalor.raws[0].get_data(),
        atol=1e-10,
    ),
)

## What this notebook did not do

- It did not guess units (except the explicit `uV` plotting argument).
- It did not resample 64 Hz EEG onto the 128 Hz envelope.
- It did not glue 20 audiobook trials into one fake continuous take.
- It did not run a TRF. That is analysis, after you have CND in MNE.

On your own lab folder the calls are the same:

```python
rec = read_cnd_mne("/path/to/dataCND", subject=1, neural_unit="uV")
```
